# Projeto Lakehouse com Apache Iceberg

## Objetivo

Demonstrar a utilização do Apache Iceberg em ambiente PySpark, utilizando dados do Campeonato Brasileiro.

## Etapas Executadas

- Inicialização SparkSession com Iceberg
- Criação do catálogo local
- Criação de namespace
- Carga do CSV
- Persistência da tabela Iceberg
- Consulta dos dados

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("IcebergProject") \
    .config(
        "spark.jars.packages",
        "org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.10.0"
    ) \
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
    ) \
    .config(
        "spark.sql.catalog.local",
        "org.apache.iceberg.spark.SparkCatalog"
    ) \
    .config(
        "spark.sql.catalog.local.type",
        "hadoop"
    ) \
    .config(
        "spark.sql.catalog.local.warehouse",
        "../data/iceberg"
    ) \
    .getOrCreate()

print("Spark Iceberg iniciado!")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/28 21:13:36 WARN Utils: Your hostname, MacBook-Air-de-Raul.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.205 instead (on interface en0)
26/04/28 21:13:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Users/castroderaul/Library/Caches/pypoetry/virtualenvs/projeto-lakehouse-JYffaGe1-py3.11/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/castroderaul/.ivy2.5.2/cache
The jars for the packages stored in: /Users/castroderaul/.ivy2.5.2/jars
org.apache.iceberg#iceberg-spark-runtime-4.0_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-35bf2589-c018-47c6-adb5-4b986acf7bfc;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-4.0_2.13;1.10.0 in central
downloading https://repo1.maven.o

Spark Iceberg iniciado!


In [2]:
df_raw = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("../data/raw/brasileirao.csv")

df_raw.show(5)

+--------------+----------+------+--------------------+-------+-------+-----------+-------------+--------------+----------------+-----------------+------------------+-------------------+-----------------------------+------------------------------+----------------------------+-----------------------------+-------------+--------------+---------------------+----------------------+-------------------+--------------------+---------------+----------------+---------------------------+----------------------------+----------------+-----------------+---------------------+----------------------+---------------+----------------+--------------------+---------------------+
|ano_campeonato|      data|rodada|             estadio|arbitro|publico|publico_max|time_mandante|time_visitante|tecnico_mandante|tecnico_visitante|colocacao_mandante|colocacao_visitante|valor_equipe_titular_mandante|valor_equipe_titular_visitante|idade_media_titular_mandante|idade_media_titular_visitante|gols_mandante|gols_visitan

26/04/28 21:14:21 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [3]:
df_raw.writeTo("local.brasileirao").using("iceberg").createOrReplace()

26/04/28 21:14:27 WARN HadoopTableOperations: Error reading version hint file ../data/iceberg/brasileirao/metadata/version-hint.text
java.io.FileNotFoundException: File ../data/iceberg/brasileirao/metadata/version-hint.text does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.hadoop.fs.ChecksumFileSystem$ChecksumFSInputChecker.<init>(ChecksumFileSystem.java:201)
	at org.apache.hadoop.fs.ChecksumFileSystem.open(ChecksumFileSystem.java:631)
	at org.apache.hadoop.fs.FileSystem.open(FileSystem.java:997)
	at org.apache.iceberg.hadoop.HadoopTableOperations.findVersion(HadoopTableOperations.java:318)
	at org.apache.iceberg.hadoop.HadoopTableOperations.ref

In [6]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.default")

DataFrame[]

In [7]:
spark.sql("SHOW NAMESPACES IN local").show()

+---------+
|namespace|
+---------+
|  default|
+---------+



In [8]:
df_raw.writeTo("local.default.brasileirao") \
    .using("iceberg") \
    .createOrReplace()

26/04/28 21:16:39 WARN HadoopTableOperations: Error reading version hint file ../data/iceberg/default/brasileirao/metadata/version-hint.text
java.io.FileNotFoundException: File ../data/iceberg/default/brasileirao/metadata/version-hint.text does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.hadoop.fs.ChecksumFileSystem$ChecksumFSInputChecker.<init>(ChecksumFileSystem.java:201)
	at org.apache.hadoop.fs.ChecksumFileSystem.open(ChecksumFileSystem.java:631)
	at org.apache.hadoop.fs.FileSystem.open(FileSystem.java:997)
	at org.apache.iceberg.hadoop.HadoopTableOperations.findVersion(HadoopTableOperations.java:318)
	at org.apache.iceberg.hadoop.HadoopTab

In [9]:
spark.sql("SHOW TABLES IN local.default").show()

+---------+-----------+-----------+
|namespace|  tableName|isTemporary|
+---------+-----------+-----------+
|  default|brasileirao|      false|
+---------+-----------+-----------+



In [10]:
spark.table("local.default.brasileirao").show(5)

+--------------+----------+------+--------------------+-------+-------+-----------+-------------+--------------+----------------+-----------------+------------------+-------------------+-----------------------------+------------------------------+----------------------------+-----------------------------+-------------+--------------+---------------------+----------------------+-------------------+--------------------+---------------+----------------+---------------------------+----------------------------+----------------+-----------------+---------------------+----------------------+---------------+----------------+--------------------+---------------------+
|ano_campeonato|      data|rodada|             estadio|arbitro|publico|publico_max|time_mandante|time_visitante|tecnico_mandante|tecnico_visitante|colocacao_mandante|colocacao_visitante|valor_equipe_titular_mandante|valor_equipe_titular_visitante|idade_media_titular_mandante|idade_media_titular_visitante|gols_mandante|gols_visitan

## Comparação rápida: Delta Lake vs Iceberg

| Tecnologia | Destaque |
|-----------|----------|
| Delta Lake | Forte integração com Spark |
| Iceberg | Excelente compatibilidade multi-engine |